# COMP6713 — Notebook 4: RAG Pipeline

This notebook demonstrates the full **Retrieval-Augmented Generation (RAG)** pipeline:

```
Question
  │
  ▼
Query Expansion  (UMLS medical term normalisation via scispaCy)
  │
  ▼
Dense Retrieval  (ChromaDB + BGE-M3 embeddings, top-10 candidates)
  │
  ▼
Reranking        (BGE cross-encoder, keeps top-3)
  │
  ▼
LLM Generation   (DeepSeek-Chat via API)
  │
  ▼
Answer + Provenance
```

**Why RAG?**  
Large language models have a knowledge cut-off and can hallucinate.  
RAG grounds the LLM in retrieved evidence from our corpus of ~21 k QA pairs  
(PubMedQA + BioASQ + MedQA-USMLE) plus ~61 k unlabelled PubMedQA abstracts.

In [ ]:
import sys, warnings, os
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

## 1. Vector Store — Load or Build

In [ ]:
from medqa.retrieval.vectorstore import VectorStore

vs = VectorStore()
print(f"Existing index size: {vs.count()} chunks")

In [ ]:
# Only build if the index is empty
if vs.count() == 0:
    print("Index is empty — building from corpus (this takes ~30 min on CPU, ~5 min on GPU)...")
    from medqa.data.loader import load_rag_corpus
    corpus = load_rag_corpus()
    vs.build(corpus)
else:
    print(f"Index already populated with {vs.count()} chunks — skipping build.")

## 2. Retrieval Demo

Let's see which chunks are retrieved for a sample medical question before any reranking.

In [ ]:
question = "What is the first-line treatment for hypertension in elderly patients?"

# Dense retrieval: embed query and find top-10 cosine-similar chunks
candidates = vs.retrieve(question, k=10)

print(f"Query: {question}\n")
print(f"Top-{len(candidates)} retrieved chunks:")
print("-" * 70)
for i, chunk in enumerate(candidates, 1):
    print(f"[{i}] Source: {chunk['source']} | Similarity: {chunk['score']:.4f}")
    print(f"    {chunk['text'][:200]}...")
    print()

## 3. Reranking

Dense retrieval finds *broadly similar* documents.  
A **cross-encoder reranker** (BGE-reranker-v2-m3) jointly encodes the question  
and each candidate passage, producing a more accurate relevance score.  
This step is slower but significantly improves precision.

In [ ]:
from medqa.retrieval.reranker import Reranker

reranker = Reranker()
reranker.load()

# Rerank: score all 10 candidates and keep top 3
top_chunks = reranker.rerank(question, candidates, top_k=3)

print(f"Query: {question}\n")
print("Top-3 after reranking:")
print("=" * 70)
for i, chunk in enumerate(top_chunks, 1):
    print(f"[{i}] Rerank score: {chunk['rerank_score']:.4f} | Source: {chunk['source']}")
    print(f"    {chunk['text'][:300]}")
    print()

## 4. Query Expansion (UMLS)

Before retrieval, the pipeline optionally expands the query using medical  
terminology from **UMLS** (Unified Medical Language System) via scispaCy.  
This helps match synonyms — e.g. "hypertension" → "high blood pressure".

In [ ]:
from medqa.data.preprocessor import expand_query_with_umls

expanded = expand_query_with_umls(question)
print(f"Original : {question}")
print(f"Expanded : {expanded}")

## 5. Full RAG Pipeline with LLM

Now we combine everything: query expansion → retrieval → reranking → LLM generation.

> **Setup**: Set your DeepSeek API key as an environment variable before running:
> ```bash
> export DEEPSEEK_API_KEY="your-key-here"
> ```
> Or add it to a `.env` file in the project root.

In [ ]:
import os

# Load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

api_key = os.environ.get("DEEPSEEK_API_KEY", "")
if not api_key:
    print("⚠  DEEPSEEK_API_KEY not set. LLM generation will be skipped in this demo.")
    print("   Set the key and re-run cell 5 onwards to see full generation.")
else:
    print("✓ API key found.")

In [ ]:
from medqa.models.llm_qa import APILLM
from medqa.retrieval.rag_pipeline import RAGPipeline

# Instantiate pipeline (reuses the VectorStore and Reranker already loaded)
if api_key:
    llm = APILLM(model="deepseek-chat")
    pipeline = RAGPipeline(
        llm=llm,
        vector_store=vs,
        reranker=reranker,
        retrieve_k=10,
        rerank_k=3,
        use_query_expansion=True,
    )
    print("RAGPipeline ready.")
else:
    pipeline = None
    print("Skipping pipeline instantiation (no API key).")

In [ ]:
if pipeline:
    result = pipeline.query(question)

    print("=" * 70)
    print(f"Question      : {result['question']}")
    print(f"Expanded query: {result['expanded_query']}")
    print()
    print("--- Retrieved context (top 3) ---")
    print(result['context'][:600], "...")
    print()
    print("--- LLM Answer ---")
    print(result['predicted_answer'])
else:
    # Show a mock result so the notebook still illustrates the output structure
    print("(Mock output — run with API key for real generation)")
    print()
    print("Question      : What is the first-line treatment for hypertension in elderly patients?")
    print()
    print("--- LLM Answer (example) ---")
    print(
        "Based on the retrieved clinical evidence, the first-line treatment for "
        "hypertension in elderly patients typically includes thiazide diuretics, "
        "calcium channel blockers (e.g. amlodipine), or ACE inhibitors/ARBs. "
        "The JNC 8 guidelines recommend initiating pharmacological treatment "
        "when SBP ≥ 150 mmHg in patients ≥ 60 years, with a target < 150/90 mmHg."
    )

## 6. Batch Evaluation on a Small Sample

In [ ]:
from medqa.data.loader import load_all
from medqa.data.preprocessor import split_dataset

all_records = load_all()
_, test = split_dataset(all_records)

# Use a small sample for quick demo (full eval is in Notebook 05)
sample = test[:10]

print(f"Evaluating on {len(sample)} test questions...")
print()

if pipeline:
    for rec in sample[:3]:
        res = pipeline.query(rec['question'])
        print(f"Q : {rec['question'][:100]}")
        print(f"GT: {rec['answer'][:100]}")
        print(f"PR: {res['predicted_answer'][:100]}")
        print()
else:
    print("(Skipping — no API key. See Notebook 05 for full evaluation.)")

## 7. RAG Component Analysis

Let's visualise how retrieval quality changes across the pipeline stages.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Compare retrieval scores before and after reranking for the demo question
if candidates and top_chunks:
    # Scores before reranking (cosine similarity)
    pre_scores  = [c['score'] for c in candidates[:10]]
    # After reranking (cross-encoder scores, normalised for display)
    post_raw    = [c['rerank_score'] for c in top_chunks]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].bar(range(1, 11), pre_scores, color='steelblue', alpha=0.8)
    axes[0].set_title('Dense Retrieval — Cosine Similarity\n(top-10 candidates)')
    axes[0].set_xlabel('Rank'); axes[0].set_ylabel('Score')
    axes[0].set_xticks(range(1, 11))
    
    axes[1].bar(range(1, len(post_raw)+1), post_raw, color='darkorange', alpha=0.8)
    axes[1].set_title('After Reranking — Cross-Encoder Score\n(top-3 kept)')
    axes[1].set_xlabel('Rank'); axes[1].set_ylabel('Score')
    axes[1].set_xticks(range(1, len(post_raw)+1))
    
    plt.suptitle(f'Query: "{question[:60]}..."', fontsize=10)
    plt.tight_layout()
    plt.savefig('../reports/figures/rag_scores.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved → reports/figures/rag_scores.png")
else:
    print("No retrieval results to plot.")

In [ ]:
# Source distribution of retrieved chunks
from collections import Counter
import matplotlib.pyplot as plt

if candidates:
    sources = [c['source'] for c in candidates]
    counts  = Counter(sources)
    
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.keys(), counts.values(), color=['steelblue','darkorange','seagreen'])
    ax.set_title('Source distribution — top-10 retrieved chunks')
    ax.set_ylabel('Count'); ax.set_xlabel('Dataset source')
    plt.tight_layout()
    plt.savefig('../reports/figures/rag_source_dist.png', dpi=120, bbox_inches='tight')
    plt.show()

## Summary

| Stage | Model | Purpose |
|-------|-------|---------|
| Query Expansion | scispaCy + UMLS | Normalise medical terminology |
| Dense Retrieval | BGE-M3 (bi-encoder) | Fast vector similarity search over 200k+ chunks |
| Reranking | BGE-reranker-v2-m3 (cross-encoder) | Precise relevance scoring of top-10 candidates |
| Generation | DeepSeek-Chat (via API) | Synthesise grounded answer from top-3 contexts |

**Key insight**: The two-stage retrieve-then-rerank approach balances speed  
(fast bi-encoder for candidate generation) with accuracy (slow cross-encoder  
only on the shortlist of 10).  
Reranking consistently promotes passages with more clinically relevant details  
to the top positions.